In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os
import random
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision import transforms

import timm

In [3]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [4]:
!pip install pytorch-metric-learning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.9 MB/s eta 0:00:0000:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.

In [5]:
from pytorch_metric_learning.samplers import MPerClassSampler
from pytorch_metric_learning import miners, losses

In [6]:
class ClothingDataset(Dataset):

    def __init__(self, samples, transform=None):

        self.samples = samples
        self.transform = transform

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        img_path, label = self.samples[idx]

        img = Image.open(
            img_path
        ).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label

In [7]:
def collect_samples(root_dir):

    samples = []

    label_to_idx = {}

    current_label = 0

    for top_folder in os.listdir(root_dir):

        top_path = os.path.join(
            root_dir,
            top_folder
        )

        if not os.path.isdir(top_path):
            continue

        for class_name in os.listdir(top_path):

            class_path = os.path.join(
                top_path,
                class_name
            )

            if not os.path.isdir(class_path):
                continue

            class_id = (
                top_folder
                + "_"
                + class_name
            )

            if class_id not in label_to_idx:

                label_to_idx[
                    class_id
                ] = current_label

                current_label += 1

            label = label_to_idx[
                class_id
            ]

            for img_name in os.listdir(class_path):

                img_path = os.path.join(
                    class_path,
                    img_name
                )

                samples.append(
                    (img_path, label)
                )

    return samples

In [8]:
all_samples = collect_samples(
    "/kaggle/input/datasets/afqwe0/clotheslabelled"
)

In [9]:
from sklearn.model_selection import train_test_split
from collections import defaultdict

In [10]:
class_samples = defaultdict(list)

for path, label in all_samples:

    class_samples[label].append(
        (path, label)
    )

In [11]:
train_samples = []
val_samples = []
test_samples = []

for label in class_samples:

    images = class_samples[label]

    train, temp = train_test_split(
        images,
        test_size=0.30,
        random_state=42
    )

    val, test = train_test_split(
        temp,
        test_size=0.50,
        random_state=42
    )

    train_samples.extend(train)
    val_samples.extend(val)
    test_samples.extend(test)

In [19]:
train_dataset = ClothingDataset(
    train_samples,
    transform
)

val_dataset = ClothingDataset(
    val_samples,
    transform
)

test_dataset = ClothingDataset(
    test_samples,
    transform
)

In [20]:
train_labels = [

    sample[1]

    for sample in train_samples
]
train_sampler = MPerClassSampler(

    train_labels,
    m=4,
    batch_size=32,
    length_before_new_iter=len(train_dataset)
)

In [21]:
train_loader = DataLoader(

    train_dataset,

    batch_size=32,

    sampler=train_sampler
)

In [22]:
val_loader = DataLoader(

    val_dataset,

    batch_size=32,

    shuffle=False
)

test_loader = DataLoader(

    test_dataset,

    batch_size=32,

    shuffle=False
)

In [23]:
def evaluate(model, loader):

    model.eval()

    embeddings = []
    labels = []

    with torch.no_grad():

        for images, batch_labels in loader:

            images = images.to(device)

            emb = model(images)

            embeddings.append(
                emb.cpu()
            )

            labels.append(
                batch_labels
            )

    embeddings = torch.cat(
        embeddings
    )

    labels = torch.cat(
        labels
    )

    similarity = torch.matmul(

        embeddings,

        embeddings.T
    )

    # remove self-match
    similarity.fill_diagonal_(
        -999
    )

    nearest = torch.argmax(
        similarity,
        dim=1
    )

    predicted = labels[
        nearest
    ]

    accuracy = (

        predicted == labels
    ).float().mean()

    return accuracy.item()

In [24]:
class MobileNetV4Embedding(
    nn.Module
):
    def __init__(
        self,
        embedding_dim=128
    ):
        super().__init__()

        self.backbone = timm.create_model(
            "mobilenetv4_conv_large.e500_r256_in1k",
            pretrained=True,
            num_classes=0
        )

        self.embedding = nn.Linear(
            1280,
            embedding_dim
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.embedding(x)

        x = F.normalize(
            x,
            p=2,
            dim=1
        )

        return x

In [25]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MobileNetV4Embedding().to(device)

In [26]:
miner = miners.TripletMarginMiner(
    margin=0.4,
    type_of_triplets="semihard"
)

loss_func = losses.TripletMarginLoss(
    margin=0.4
)

In [27]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [28]:
from tqdm.auto import tqdm

best_val_acc = 0

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    progress_bar = tqdm(

        train_loader,

        desc=f"Epoch {epoch+1}/{epochs}"
    )

    for images, labels in progress_bar:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        embeddings = model(
            images
        )

        hard_triplets = miner(

            embeddings,

            labels
        )

        loss = loss_func(

            embeddings,

            labels,

            hard_triplets
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        avg_loss = (

            total_loss /

            (progress_bar.n + 1)
        )

        progress_bar.set_postfix({

            "loss":
            f"{avg_loss:.4f}"
        })

    epoch_loss = (

        total_loss /

        len(train_loader)
    )

    print(
        f"Epoch {epoch+1}"
        f" | Train Loss {epoch_loss:.4f}"
    )

    # ---------- validation ----------

    val_acc = evaluate(
        model,
        val_loader
    )

    print(
        f"Validation Accuracy:"
        f" {val_acc:.4f}"
    )

    # save best model

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(

            model.state_dict(),

            "best_model.pth"
        )

        print(
            "Saved best model"
        )

Epoch 1/30:   0%|          | 0/65 [00:00<?, ?it/s]

Epoch 1 | Train Loss 0.2253
Validation Accuracy: 0.4978
Saved best model


Epoch 2/30:   0%|          | 0/65 [00:00<?, ?it/s]

Epoch 2 | Train Loss 0.1782
Validation Accuracy: 0.6422
Saved best model


Epoch 3/30:   0%|          | 0/65 [00:00<?, ?it/s]

...

Epoch 28 | Train Loss 0.0623
Validation Accuracy: 0.7978


Epoch 29/30:   0%|          | 0/65 [00:00<?, ?it/s]

Epoch 29 | Train Loss 0.0577
Validation Accuracy: 0.8022


Epoch 30/30:   0%|          | 0/65 [00:00<?, ?it/s]

Epoch 30 | Train Loss 0.0618
Validation Accuracy: 0.8244


In [29]:
model.load_state_dict(

    torch.load(
        "best_model.pth"
    )
)

<All keys matched successfully>

In [31]:
test_acc = evaluate(

    model,

    test_loader
)

print(
    "Final Test Accuracy:",
    test_acc
)

Final Test Accuracy: 0.7874186635017395
